In [20]:
pip install requests beautifulsoup4 pdfplumber pandas

Note: you may need to restart the kernel to use updated packages.


In [7]:
#!/usr/bin/env python3
"""
Combined MA House RollCalls scraper (2001-2025)

- Old pattern (2001-2010/2011): /Journal/House/<session>/RollCalls/hjMMDDYYYY_RollCalls.pdf
- New pattern (2011-2025): /Journal/House/<session>/<year>/RollCalls  (may be PDF or HTML index)
Saves files to ./HJRollCalls/<year>/
"""

import requests
import os
import time
import random
import traceback
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse


base = "https://malegislature.gov/Journal/House"
size = 2048
delay_min = 0.2
delay_max = 0.8
save_root = "./HJRollCalls"
request_timeout = 20
max_retries = 2

#Explicit session (years mapping) 
session_years = {
    182: [2001, 2002],
    183: [2003, 2004],
    184: [2005, 2006],
    185: [2007, 2008],
    186: [2009, 2010],
    187: [2011, 2012],
    188: [2013, 2014],
    189: [2015, 2016],
    190: [2017, 2018],
    191: [2019, 2020],
    192: [2021, 2022],
    193: [2023, 2024],
    194: [2025],
}

#days/months for brute forcing older pattern
months = list(range(1, 13))
days = list(range(1, 32))


def safe_get(url, stream=False, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES):
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, stream=stream, timeout=timeout)
            return r
        except Exception as e:
            if attempt < retries:
                time.sleep(0.5 + attempt * 0.5)
                continue
            return None


def is_pdf_response(resp):
    if not resp:
        return False
    ct = resp.headers.get("Content-Type", "")
    return "pdf" in ct.lower()


def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_response_pdf(resp, save_path):
    ensure_dir(os.path.dirname(save_path))
    with open(save_path, "wb") as f:
        if hasattr(resp, "iter_content"):
            for i in resp.iter_content(i=size):
                if i:
                    f.write(i)
        else:
            f.write(resp.content)


def make_abs(link, base_url):
    return urljoin(base_url, link)


def extract_pdf_links_from_html(html_text, base_url):
    soup = BeautifulSoup(html_text, "html.parser")
    links = set()

    # standard anchors
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.lower().endswith(".pdf"):
            links.add(make_abs(href, base_url))

    # look for url containing .pdf
    for tag in soup.find_all():
        for attr in ("data-href", "data-url", "data-download", "data-file"):
            val = tag.attrs.get(attr)
            if isinstance(val, str) and val.lower().endswith(".pdf"):
                links.add(make_abs(val, base_url))
        onclick = tag.attrs.get("onclick")
        if onclick and ".pdf" in onclick:
            # crude parse: split by quotes and pick pieces containing .pdf
            parts = onclick.replace("'", '"').split('"')
            for p in parts:
                if ".pdf" in p:
                    links.add(make_abs(p, base_url))

    return list(links)

def fetch_rollcalls_for_session_year(session, year, use_delay=False):
    """Try multiple strategies to retrieve rollcall PDFs for one session-year."""
    saved_files = []
    target_dir = os.path.join(save_root, str(year))
    ensure_dir(target_dir)

    base_index_url = f"{base}/{session}/{year}/RollCalls"
    candidate_urls = [
        base_index_url,            # may be pdf or html index
        base_index_url + ".pdf",   
    ]

    #try the candidate urls
    for url in candidate_urls:
        if use_delay:
            time.sleep(random.uniform(delay_min , delay_max))
        print("Trying:", url)
        resp = safe_get(url, stream=True)
        if not resp:
            print("  -> request failed")
            continue

        #if response is PDF: save it
        if is_pdf_response(resp):
            fname = f"{session}_{year}_RollCalls.pdf"
            save_path = os.path.join(target_dir, fname)
            if os.path.exists(save_path):
                print("already saved:", save_path)
            else:
                print("PDF found:", save_path)
                save_response_pdf(resp, save_path)
            saved_files.append(save_path)
            #don't return yet ans also attempt to parse HTML in case there are separate PDFs
            continue

        #if HTML, parse for explicit pdf links
        ct = resp.headers.get("Content-Type", "")
        if "html" in ct.lower() or resp.text:
            print("HTML index detected; extracting .pdf links")
            links = extract_pdf_links_from_html(resp.text, base_index_url)
            print(f"{len(links)} candidate pdf links found")
            for pdf_link in links:
                if use_delay:
                    time.sleep(random.uniform(delay_min,delay_max))
                try:
                    rpdf = safe_get(pdf_link, stream=True)
                    if rpdf and is_pdf_response(rpdf):
                        fname = os.path.basename(urlparse(pdf_link).path)
                        save_path = os.path.join(target_dir, fname)
                        if os.path.exists(save_path):
                            print("already exists:", save_path)
                            saved_files.append(save_path)
                            continue
                        print(" downloading:", pdf_link)
                        save_response_pdf(rpdf, save_path)
                        saved_files.append(save_path)
                    else:
                        print("link isn't a pdf or failed:", pdf_link)
                except Exception as ex:
                    print("error:", ex)
            

    #try folder path with trailing slash
    folder_url = f"{base}/{session}/{year}/RollCalls/"
    try:
        if use_delay:
            time.sleep(random.uniform(delay_min, delay_max))
        r = safe_get(folder_url)
        if r and r.status_code == 200 and ("html" in r.headers.get("Content-Type", "").lower() or r.text):
            more_links = extract_pdf_links_from_html(r.text, folder_url)
            for pdf_link in more_links:
                if pdf_link in saved_files:
                    continue
                if use_delay:
                    time.sleep(random.uniform(delay_min,delay_max))
                rpdf = safe_get(pdf_link, stream=True)
                if rpdf and is_pdf_response(rpdf):
                    fname = os.path.basename(urlparse(pdf_link).path)
                    save_path = os.path.join(target_dir, fname)
                    if not os.path.exists(save_path):
                        save_response_pdf(rpdf, save_path)
                        saved_files.append(save_path)
    except Exception:
        pass

    #If year is old-style (before 2011 or soo on)

    if year <= 2011 and len(saved_files) == 0:
        print("No PDFs found by index; trying date-based pattern for year", year)
        for month in months:
            for day in days:
                if use_delay:
                    time.sleep(random.uniform(delay_min, delay_max))
                date_code = f"hj{month:02d}{day:02d}{year}"
                pattern_url = f"{BASE}/{session}/RollCalls/{date_code}_RollCalls.pdf"
                r = safe_get(pattern_url, stream=True)
                if r and is_pdf_response(r):
                    save_path = os.path.join(target_dir, f"{date_code}.pdf")
                    if os.path.exists(save_path):
                        print("  -> already exists:", save_path)
                        saved_files.append(save_path)
                        continue
                    print("  -> found date-based pdf:", pattern_url)
                    save_response_pdf(r, save_path)
                    saved_files.append(save_path)

    return saved_files


def scrape_all(use_delay=False):
    saved_all = []
    for session, years in session_years.items():
        for year in years:
            print("\n=== Session", session, "Year", year, "===")
            try:
                found = fetch_rollcalls_for_session_year(session, year, use_delay=use_delay)
                print(f"Saved {len(found)} files for session {session} year {year}")
                saved_all.extend(found)
            except Exception:
                print("Error scraping session/year:", session, year)
                traceback.print_exc()
    print("\nDone. Total files saved:", len(saved_all))
    return saved_all

if __name__ == "__main__":
    # Set use_delay=True in polite mode
    files = scrape_all(use_delay=True)



=== Session 182 Year 2001 ===
Trying: https://malegislature.gov/Journal/House/182/2001/RollCalls
  -> request failed
Trying: https://malegislature.gov/Journal/House/182/2001/RollCalls.pdf
  -> request failed
No PDFs found by index; trying date-based pattern for year 2001
  -> already exists: ./HJRollCalls/2001/hj12052001.pdf
Saved 1 files for session 182 year 2001

=== Session 182 Year 2002 ===
Trying: https://malegislature.gov/Journal/House/182/2002/RollCalls
  -> request failed
Trying: https://malegislature.gov/Journal/House/182/2002/RollCalls.pdf
  -> request failed
No PDFs found by index; trying date-based pattern for year 2002
  -> already exists: ./HJRollCalls/2002/hj01022002.pdf
  -> already exists: ./HJRollCalls/2002/hj01032002.pdf
  -> already exists: ./HJRollCalls/2002/hj01162002.pdf
  -> already exists: ./HJRollCalls/2002/hj01222002.pdf
  -> already exists: ./HJRollCalls/2002/hj01302002.pdf
  -> already exists: ./HJRollCalls/2002/hj02062002.pdf
  -> already exists: ./HJRoll